In [ ]:
# Cài đặt các thư viện cốt lõi của LangChain và các gói tích hợp
!pip install langchain langchain-community langchain-huggingface

# Cài đặt FAISS để làm vector store (chạy trên CPU)
!pip install faiss-cpu

# Cài đặt thư viện để tải model embedding từ Hugging Face
!pip install sentence-transformers

# Cài đặt thư viện để gọi API của Hugging Face (cho LLM)
!pip install huggingface_hub

# Cài đặt thư viện để đọc file .txt (đã có sẵn) hoặc .pdf
!pip install pypdf

# # --- 1. CÀI ĐẶT THƯ VIỆN ---
# # (Đã cập nhật để sửa lỗi 'InferenceClient' object has no attribute 'post')

# print("Đang cài đặt/nâng cấp các thư viện...")

# # --- GIẢI PHÁP ---
# # Nâng cấp các gói LangChain và Hugging Face Hub lên phiên bản mới nhất
# # để đảm bảo chúng tương thích với nhau (sửa lỗi .post())
# !pip install --upgrade langchain langchain-community langchain-huggingface langchain-text-splitters huggingface_hub

# # Cài đặt các thư viện khác (giữ nguyên)
# !pip install faiss-cpu
# !pip install sentence-transformers
# !pip install pypdf

# print("✅ Đã cài đặt/nâng cấp xong!")
# print("\n--- QUAN TRỌNG ---")
# print("Vui lòng 'Khởi động lại phiên' (Restart session) trong menu 'Thời gian chạy' (Runtime)")
# print("trước khi chạy lại các cell khác.")



  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_huggingface-1.0.0-py3-none-any.whl.metadata (2.1 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of

In [ ]:
# --- Import các thư viện ---
from langchain_community.document_loaders import PyPDFLoader # <-- ĐÃ THAY ĐỔI
from google.colab import files # <-- ĐÃ THÊM
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFaceHub
from langchain.chains import RetrievalQA # <-- ĐÃ THAY ĐỔI

print(" Tất cả thư viện đã được import.")

✅ Tất cả thư viện đã được import.


In [ ]:
# ===  KẾT NỐI GOOGLE DRIVE ===
print(" Đang kết nối Google Drive...")
drive.mount('/content/drive')

# ===  CHỈ ĐỊNH ĐƯỜNG DẪN FILE PDF TRONG DRIVE ===
# Hãy sửa đường dẫn bên dưới theo vị trí file thật trong Drive của bạn
pdf_path = "/content/drive/MyDrive/Colab Notebooks/LangChain/Stress.pdf"
#  Ví dụ:
# Nếu file nằm trong thư mục con 'AI_Project' của MyDrive:
# pdf_path = "/content/drive/MyDrive/AI_Project/Stress.pdf"

# ===  KIỂM TRA FILE TỒN TẠI ===
if not os.path.exists(pdf_path):
    print(f"\n Lỗi: Không tìm thấy file tại '{pdf_path}'.")
    print(" Vui lòng kiểm tra lại đường dẫn hoặc tên file (chú ý chữ hoa/thường).")
    chunks = []  # Tránh lỗi ở cell sau
else:
    # ===  TẢI FILE PDF ===
    print(f"\n Đang tải tài liệu từ: {pdf_path}")
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    print(f" Đã tải {len(docs)} trang từ PDF.")

    # ===  CHIA TÀI LIỆU THÀNH CÁC CHUNK ===
    print("✂️ Đang chia tài liệu...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks = text_splitter.split_documents(docs)

    # ===  THÔNG BÁO KẾT QUẢ ===
    print(f"📄 Đã chia tài liệu thành {len(chunks)} chunks.")

🔗 Đang kết nối Google Drive...
Mounted at /content/drive

📥 Đang tải tài liệu từ: /content/drive/MyDrive/Colab Notebooks/LangChain/Stress.pdf
✅ Đã tải 6 trang từ PDF.
✂️ Đang chia tài liệu...
📄 Đã chia tài liệu thành 22 chunks.


In [ ]:
# --- 5. VECTOR HÓA VÀ LƯU (Embed & Store) ---

# === YÊU CẦU 4: Vector hóa và Lưu (Embed & Store) ===
print("Đang tải model embedding local (VoVanPhuc)...")
# Dùng model embedding Tiếng Việt (chạy local, miễn phí)
model_name = "VoVanPhuc/sup-SimCSE-VietNamese-phobert-base"
embeddings = HuggingFaceEmbeddings(model_name=model_name)
print("Đã tải model embedding.")

print("Đang vector hóa các chunk và lưu vào FAISS...")
# Tạo vector store
db = FAISS.from_documents(chunks, embeddings)
print(" Đã tạo Vector Store FAISS!")


Đang tải model embedding local (VoVanPhuc)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Đã tải model embedding.
Đang vector hóa các chunk và lưu vào FAISS...
✅ Đã tạo Vector Store FAISS!


In [ ]:
# --- 6. TẠO RAG CHAIN (LLM + Retriever) ---

# === YÊU CẦU 5: Tạo RetrievalQA chain ===
print("Đang tải model về local thay vì dùng Endpoint...")

# GIẢI PHÁP: Load model về local thay vì dùng HuggingFaceEndpoint
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFacePipeline

# Tải model và tokenizer
model_name = "bigscience/mt0-base"
print(f"Đang tải model {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Tạo pipeline
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    device=-1  # CPU, đổi thành 0 nếu có GPU
)

# Tạo LLM từ pipeline
llm = HuggingFacePipeline(pipeline=pipe)
print(" Đã tải model local!")

# 5b. Tạo Retriever (Người tìm tài liệu)
# k=2: Lấy 2 chunk liên quan nhất
retriever = db.as_retriever(search_kwargs={"k": 2})

# 5c. Tạo Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True # Yêu cầu trả về nguồn
)
print(" Đã tạo RAG Chain!")

Đang tải model về local thay vì dùng Endpoint...
Đang tải model bigscience/mt0-base...


tokenizer_config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/798 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Đã tải model local!
✅ Đã tạo RAG Chain!


In [ ]:
# --- 7. HỎI CHAIN VÀ IN KẾT QUẢ ---

# === YÊU CẦU 6: Hỏi Chain ===
query = "Hãy nêu 2 kỹ thuật hít thở được đề cập trong tài liệu?"
print(f"\nĐang thực hiện truy vấn: {query}\n")

result = qa_chain.invoke({"query": query})

# === YÊU CẦU 7: Phân tích & Kiểm chứng ===
print("--- KẾT QUẢ TRUY VẤN ---")

# In câu trả lời do LLM (mt0-base) tạo ra
print("\n[💬 Câu trả lời của LLM]:")
print(result["result"])

# In các chunk nguồn mà Retriever tìm thấy
print("\n\n[📚 Nguồn tài liệu (Kiểm chứng)]:")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n--- Nguồn {i} ---")
    print(doc.page_content)
    print("------------------")


Đang thực hiện truy vấn: Hãy nêu 2 kỹ thuật hít thở được đề cập trong tài liệu?

--- KẾT QUẢ TRUY VẤN ---

[💬 Câu trả lời của LLM]:
Thở bằng bụng, thở theo nhịp 4 – 7 – 8


[📚 Nguồn tài liệu (Kiểm chứng)]:

--- Nguồn 1 ---
thở. 
 Để ý xem mình cảm thấy thế nào khi kết thúc bài tập. 
2.2. Các bài tập thở nâng cao 
Sau khi đã thành thạo kỹ thuật thở bụng, bạn có thể muốn thử một trong những 
bài tập thở nâng cao hơn này. Lời khuyên là hãy thử cả ba và xem cái nào phù 
hợp nhất với mình: 
 Thở theo nhịp 4 – 7 – 8 
 Thở cuộn 
 Thở buổi sáng
------------------

--- Nguồn 2 ---
thở sâu để giảm cảm giác lo lắng cũng như hạn chế tiêu hao năng lượng một 
cách không cần thiết. Đối với những người thường xuyên gặp phải những áp 
lực trong cuộc sống, họ có thể thử áp dụng một số kỹ thuật thở như : Thở 
bằng bụng, thở theo nhịp 4 – 7 – 8, thở cuộn, thực hiện các bài tập thở vào 
buổi sáng... để giúp cải thiện tâm trạng cũng như nâng cao sức khỏe nói 
chung.
------------------
